# EIA Testing

The purpose of this notebook is to test the ingestion process from EIA API  

In [1]:
import requests

import pandas as pd
from datetime import datetime
import duckdb
import pyarrow as pa

import os
from dotenv import load_dotenv

from sqlalchemy import (
    Column, 
    Table, 
    MetaData, 
    text, 
    Text, 
    Numeric, 
    Integer, 
    BigInteger, 
    Boolean, 
    DateTime, 
    create_engine, 
    inspect
)

from sqlalchemy.dialects.postgresql import insert as pg_insert

import json

In [2]:
TYPE_MAP = {
    'TEXT' : Text,
    'NUMERIC' : Numeric,
    'INTEGER' : Integer,
    'BIGINT' : BigInteger,
    'BOOLEAN' : Boolean,
    'TIMESTAMP' : DateTime,
}

In [3]:
load_dotenv()

API_KEY = os.getenv("EIA_API_KEY")

# Consider adding a try-except block to handle potential errors in the API request and missing API keys

In [4]:
BASE_URL = "https://api.eia.gov/v2"

In [32]:
def eia_get_facet_values(route: str, facet: str) -> pd.DataFrame:
    """
    Query the EIA API facet endpoint to get all valid values for a given facet.
    This is the authoritative way to discover valid codes before building params.
    """
    url = f"{BASE_URL}/{route}/facet/{facet}/"
    r = requests.get(url, params={"api_key": API_KEY}, timeout=30)
    r.raise_for_status()
    
    facets_list = r.json()['response']['facets']
    facets_arrow = pa.Table.from_pylist(facets_list)
    return duckdb.sql("SELECT * FROM facets_arrow")

In [31]:
def eia_build_params(
        route: str,
        frequency: str = "monthly",
        start: str = "2000-01",
        end: str = "2025-12",
        facets: dict = None, # { "facet_name": [list of ids] or None for all }
        data_cols: list = None
) -> list:
    """
    Resolves facet values and buildes a list of tuples

    Parameters:
    - route: The API endpoint route (e.g., 'series')
    - frequency: Data frequency (e.g., 'monthly')
    - start: Start date in YYYY-MM format
    - end: End date in YYYY-MM format
    - facets: { "facet_name": [list of ids] or None for auto-discovery all }
    - data_cols: List of data columns to retrieve (e.g., ['value', 'units']). Defaults to ['value'] if None.

    Returns:
    A list of tuples for eia_fetch_all()
    """

    data_cols = data_cols or ["value"]

    # --- Base params ---
    params = [
        ("frequency",           frequency),
        ("start",               start),
        ("end",                 end),
        ("sort[0][column]",     "period"),
        ("sort[0][direction]",  "asc")
    ]

    # --- Add data columns dynamically ---  
    for i, col in enumerate(data_cols):
        params.append((f"data[{i}]", col))

    # --- Auto-discover facet values if not provided --- 
    resolved_facets = {}
    for facet_name, facet_values in (facets or {}).items():
        if facet_values is None:
            facets_rel = eia_get_facet_values(route, facet_name)
            resolved_facets[facet_name] = [row[0] for row in facets_rel.fetchall()]
            print(f"[INFO] Discovered {len(resolved_facets[facet_name])} values for facet '{facet_name}': {resolved_facets[facet_name]}")
        else:
            resolved_facets[facet_name] = facet_values
            print(f"[INFO] Using provided values for facet '{facet_name}': {resolved_facets[facet_name]}")

    # --- Add facet values to params ---
    for facet_name, facet_values in resolved_facets.items():
        for val in facet_values:
            params.append((f"facets[{facet_name}][]", val))

    return params

In [7]:
"""
The logic: instead of sending all 41 duoarea values in one request, 
it splits them into batches of 10, makes 5 separate requests (4 × 10 + 1 × 1), 
paginates each one fully, then combines everything into a single relation at the end. 
The chunk_size=10 is conservative — you can bump it to 15 if you want fewer requests, 
but 10 keeps you safely under URL length limits even with multiple facet types combined.
"""

def eia_fetch_all(
            route: str, 
            params: list,
            chunk_size: int = 10 # max facet values per request
    ) -> duckdb.DuckDBPyRelation:
    """
    Paginated fetch with facet chunking to avoid URL length limits.
    Splits large facet lists into smaller batches and combines results.

    Parameters:
    - route: The API endpoint route (e.g., 'series')
    - params: A dictionary of parameters to include in the API request

    Returns:
    - A DuckDB relation of all pages combined
    """

    # --- Separate facet parms from base params --- 
    base_params = [(k,v) for k,v in params if "facets" not in k]
    facet_params = [(k,v) for k,v in params if "facets" in k]

    # --- Group facets by name ---
    # e.g. { "duoarea": ["STX", "SNM", ...], "product": ["EPC0", ...]}
    facet_groups = {}
    for k,v in facet_params:
        facet_groups.setdefault(k, []).append(v)

    # --- Chunk only the largest facet group to keep URLs short ---
    # Find the facet with the most values to chunk on
    primary_facet_key = max(facet_groups, key=lambda k: len(facet_groups[k]))
    primary_values = facet_groups[primary_facet_key]
    other_facets = [(k,v) for k, vals in facet_groups.items()
                    if k != primary_facet_key for v in vals]
    
    chunks = [
        primary_values[i:i + chunk_size]
        for i in range(0, len(primary_values), chunk_size)
    ]

    print(f"[INFO] Chunking '{primary_facet_key}' into "
          f"{len(chunks)} batches of {chunk_size}")

    all_records = []

    for chunk_idx, chunk in enumerate(chunks):

        chunk_facet_params = [(primary_facet_key, v) for v in chunk]
        offset = 0
        page_size = 5000

        while True:

            paged_params = (
                base_params
                + chunk_facet_params
                + other_facets
                + [
                ("offset", offset),
                ("length", page_size),
                ("api_key", API_KEY)
                ]
            )

            url = f"{BASE_URL}/{route}/data/"
        
            try:
                response = requests.get(url, params=paged_params, timeout=30)
                response.raise_for_status()  # Raise an error for bad status codes
                
                payload = response.json()['response']      
                records = payload['data']
                total = int(payload.get("total", 0))

                all_records.extend(records)
                offset += page_size

                print(f"[INFO] Chunk {chunk_idx + 1}/{len(chunks)} -"
                      f"Fetched {min(offset, total)}/{total} records from {route}")

                if offset >= total:
                    break  # Exit the loop if we've fetched all records
                
            except requests.exceptions.RequestException as e:
                print(
                    f"[ERROR] Chunk {chunk_idx + 1}/{len(chunks)} -"
                    f"failed at offset {offset}: {e}")
                break # Return nothing
    
    if (not all_records):
        return duckdb.sql("SELECT 1 WHERE FALSE")

    all_records_arrow = pa.Table.from_pylist(all_records)
    return duckdb.sql("SELECT * FROM all_records_arrow")

In [8]:
def create_postgres_engine(user: str, password: str, host: str, port: int, database: str) -> create_engine:
    """
    Create a SQLAlchemy engine for connecting to a PostgreSQL database.

    Parameters:
    - user: Database username
    - password: Database password
    - host: Database host (e.g., 'localhost')
    - port: Database port (e.g., 5432)
    - database: Database name

    Returns:
    - A SQLAlchemy engine instance
    """
    connection_string = f"postgresql://{user}:{password}@{host}:{port}/{database}"
    engine = create_engine(connection_string)
    return engine

In [9]:
def create_schema_if_not_exists(engine: create_engine, schema_name: str):
    """
    Create a schema in the PostgreSQL database if it does not already exist.

    Parameters:
    - engine: A SQLAlchemy engine instance
    - schema_name: The name of the schema to create
    """
    with engine.begin() as connection:
        connection.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema_name};"))
    print(f"Schema '{schema_name}' is ready.")

In [10]:
def write_to_postgres(
        relation: duckdb.DuckDBPyRelation, 
        table_name: str, 
        schema: dict, 
        engine: create_engine,
        pg_schema: str = "bronze"
        ) -> None: 
    """
    Dynamically create or upsert a PostgreSQL table from a DuckDB relation.

    Schema format:
        { "column_name": ("DATA_TYPE", is_primary_key) }

    Behaviour:
        - Table doesn't exist  → creates it with defined PKs, then inserts
        - Table exists         → upserts on conflict of PK columns
    """

    # --- Auto-create schema ---
    create_schema_if_not_exists(engine, pg_schema)

    metadata = MetaData(schema=pg_schema)
    inspector = inspect(engine)

    # --- Build Column objects from schema dict --- 
    columns = []
    pk_columns = []

    for col_name, (dtype, is_pk) in schema.items():
        col_type = TYPE_MAP.get(dtype.upper())
        if not col_type:
            raise ValueError(f"Unsupported data type '{dtype}' for column '{col_name}'")
        
        columns.append(Column(col_name, col_type, primary_key=is_pk))
        if is_pk:
            pk_columns.append(col_name)
        
    if not pk_columns:
        raise ValueError("At least one primary key column must be defined in the schema.")

    # --- Define table object ---
    table = Table(table_name, metadata, *columns)

    # --- Create table if it doesn't exist --- 
    existing_tables = inspector.get_table_names(schema=pg_schema)
    if table_name not in existing_tables:
        metadata.create_all(engine)
        print(f"[INFO] Created '{pg_schema}.{table_name}' with PKs: {pk_columns}")

    # --- Extract records from DuckDB relation --- 
    # Only keep columns that exist in both the schema and the relation
    available_cols = relation.columns
    selected_cols = [c for c in schema.keys() if c in available_cols]

    if (not selected_cols):
        raise ValueError("No overlapping columns between schema and relation.")

    # Project down to only the columns we need, then pull as list of dicts
    col_list = ", ".join(f'"{c}"' for c in selected_cols)
    pk_list = ", ".join(f'"{c}"' for c in pk_columns)

    # --- Added deduplicate on PK columns before extracting - keeps last record per PK
    records = (
        duckdb.sql(f"""
            SELECT
            {col_list}       
            FROM (                
                SELECT 
                    {col_list}
                    ,ROW_NUMBER() OVER (
                    PARTITION BY {pk_list}
                    ORDER BY (SELECT NULL) -- arbitrary order, just keep last record per PK
                    ) AS rn
                FROM relation
            ) deduped
            WHERE rn = 1
        """)
        .fetchdf()
        .to_dict(orient="records")
    )

    if (not records):
        print(f"[WARN] No records to write to '{pg_schema}.{table_name}'.")
        return

    # --- Upsert --- 
    non_pk_cols = [c for c in selected_cols if c not in pk_columns]

    with engine.begin() as conn:
        stmt = pg_insert(table).values(records)

        if non_pk_cols:
            upsert_stmt = stmt.on_conflict_do_update(
                index_elements=pk_columns,
                set_={col: stmt.excluded[col] for col in non_pk_cols}
            )
        else:
            upsert_stmt = stmt.on_conflict_do_nothing(index_elements=pk_columns)
        
        conn.execute(upsert_stmt)
    
    print(f"[INFO] Upserted {len(records)} records into '{pg_schema}.{table_name}' on PK conflict of {pk_columns}.")




In [11]:
def eia_get_all_facets(route: str) -> list:
    """Returns all available facet names for a given route."""
    url = f"{BASE_URL}/{route}/"
    r = requests.get(url, params={"api_key": API_KEY}, timeout=30)
    r.raise_for_status()
    return [f["id"] for f in r.json()["response"].get("facets", [])]

In [38]:
def eia_get_data_columns(route: str) -> list:
    """Returns valid data column names for a given route."""
    url = f"{BASE_URL}/{route}/"
    r = requests.get(url, params={"api_key": API_KEY}, timeout=30)
    r.raise_for_status()
    return list(r.json()["response"].get("data", {}).keys())
    

In [44]:
def eia_get_route_metadata(route: str) -> dict:
    """Returns metadata for a given route - facets, data columns, frequencies"""
    url = f"{BASE_URL}/{route}/"
    r = requests.get(url, params={"api_key": API_KEY}, timeout=30)
    r.raise_for_status()
    response = r.json()["response"]
    return {
        "facets":       response.get("facets", []),
        "data_cols":    list(response.get("data", {}).keys()),
        "frequencies":  response.get("frequencies", [])
    }

### Execute

In [13]:
# Create the engine for PostgreSQL connection
engine = create_postgres_engine(
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    database=os.getenv("POSTGRES_DB")
)

In [12]:
for route in [
        "petroleum/crd/crpdn",
        "petroleum/sum/snd", 
        "petroleum/sum/sndw", 
        "natural-gas/prod/ngpl", 
        "natural-gas/prod/sum"
    ]:
    facets = eia_get_all_facets(route)
    print(f"{route}: {facets}")

petroleum/crd/crpdn: ['duoarea', 'product', 'process', 'series']
petroleum/sum/snd: ['duoarea', 'product', 'process', 'series']
petroleum/sum/sndw: ['duoarea', 'product', 'process', 'series']
natural-gas/prod/ngpl: ['duoarea', 'product', 'process', 'series']
natural-gas/prod/sum: ['duoarea', 'product', 'process', 'series']


#### petroleum/crd/crpdn

(Crude Oil Production)

In [14]:
params = eia_build_params(
    route="petroleum/crd/crpdn",
    frequency="monthly",
    start="2000-01",
    end="2025-12",
    facets={
        "duoarea": None,
        "product": None,
        "process": None,
        "series": None
    }
)

production_rel = eia_fetch_all("petroleum/crd/crpdn", params)


[INFO] Discovered 41 values for facet 'duoarea': ['SNE', 'SNY', 'SOH', 'R20', 'SID', 'SIL', 'SCA', 'SND', 'SNM', 'SNV', 'SUT', 'SWY', 'RAKS', 'SKS', 'SKY', 'R30', 'SSD', 'SVA', 'SAL', 'SAR', 'SFL', 'SMS', 'R10', 'SWV', 'SMI', 'SAK', 'R3FM', 'SCO', 'SMT', 'SOK', 'R50', 'STN', 'R5F', 'R40', 'NUS', 'SAZ', 'SIN', 'SLA', 'SMO', 'SPA', 'STX']
[INFO] Chunking 'facets[duoarea][]' into 5 batches of 10
[INFO] Chunk 1/5 -Fetched 5000/6072 records from petroleum/crd/crpdn
[INFO] Chunk 1/5 -Fetched 6072/6072 records from petroleum/crd/crpdn
[INFO] Chunk 2/5 -Fetched 5000/6240 records from petroleum/crd/crpdn
[INFO] Chunk 2/5 -Fetched 6240/6240 records from petroleum/crd/crpdn
[INFO] Chunk 3/5 -Fetched 5000/6864 records from petroleum/crd/crpdn
[INFO] Chunk 3/5 -Fetched 6864/6864 records from petroleum/crd/crpdn
[INFO] Chunk 4/5 -Fetched 5000/6240 records from petroleum/crd/crpdn
[INFO] Chunk 4/5 -Fetched 6240/6240 records from petroleum/crd/crpdn
[INFO] Chunk 5/5 -Fetched 624/624 records from petro

In [15]:
print(production_rel.describe())

┌─────────┬─────────┬─────────┬────────────┬─────────┬───────────────┬─────────┬──────────────────┬──────────────────────┬──────────────────────────────────────────────────────────────────┬─────────┬─────────┐
│  aggr   │ period  │ duoarea │ area-name  │ product │ product-name  │ process │   process-name   │        series        │                        series-description                        │  value  │  units  │
│ varchar │ varchar │ varchar │  varchar   │ varchar │    varchar    │ varchar │     varchar      │       varchar        │                             varchar                              │ varchar │ varchar │
├─────────┼─────────┼─────────┼────────────┼─────────┼───────────────┼─────────┼──────────────────┼──────────────────────┼──────────────────────────────────────────────────────────────────┼─────────┼─────────┤
│ count   │ 26040   │ 26040   │ 26040      │ 26040   │ 26040         │ 26040   │ 26040            │ 26040                │ 26040                                

In [16]:
production_rel = duckdb.sql("""
        SELECT
            CAST(period  || '-01' AS TIMESTAMP)   AS period
            ,duoarea
            ,"area-name"                AS "area_name"
            ,product
            ,"product-name"             AS "product_name"
            ,process         
            ,"process-name"             AS "process_name"
            ,series
            ,"series-description"       AS "series_description"
            ,TRY_CAST(value AS DOUBLE)  AS value
            ,units
        FROM production_rel
    """)

In [ ]:
# Define the schema for the PostgreSQL table
write_to_postgres(
    relation=production_rel,
    table_name="t_crude_oil_production",
    schema={
        "period":               ("TIMESTAMP", True),
        "duoarea":              ("TEXT", False),
        "area_name":            ("TEXT", True),
        "product":              ("TEXT", True),
        "product_name":         ("TEXT", False),
        "process":              ("TEXT", True),
        "process_name":         ("TEXT", False),
        "series":               ("TEXT", True),
        "series_description":   ("TEXT", False),
        "value":                ("NUMERIC", False),
        "units":                ("TEXT", False)
    },
    engine=engine,
)

Schema 'bronze' is ready.
[INFO] Upserted 26028 records into 'bronze.crude_oil_production' on PK conflict of ['period', 'area_name', 'product', 'process', 'series'].


#### petroleum/sum/snd

(Supply and Disposition)

In [19]:
params = eia_build_params(
    route="petroleum/sum/snd",
    frequency="monthly",
    start="2000-01",
    end="2025-12",
    facets={
        "duoarea": None,
        "product": None,
        "process": None,
        "series": None
    }
)

bs_rel = eia_fetch_all("petroleum/sum/snd", params)


[INFO] Discovered 17 values for facet 'duoarea': ['R20', 'R30-Z0P', 'R10-Z00', 'R30-Z00', 'R50-Z00', 'R30', 'NUS-Z00', 'R10', 'R20-Z00', 'R40-Z00', 'R50-Z0P', 'R50', 'R10-Z0P', 'R40-Z0P', 'R40', 'NUS', 'R20-Z0P']
[INFO] Chunking 'facets[duoarea][]' into 2 batches of 10
[INFO] Chunk 1/2 -Fetched 5000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 10000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 15000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 20000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 25000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 30000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 35000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 40000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 45000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 50000/624415 records from petroleum/sum/snd
[INFO] Chunk 1/2 -Fetched 55000

In [21]:
print(bs_rel.describe())

┌─────────┬─────────┬─────────┬───────────┬─────────┬──────────────────────┬─────────┬───────────────────────────────┬──────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┐
│  aggr   │ period  │ duoarea │ area-name │ product │     product-name     │ process │         process-name          │        series        │                                                  series-description                                                   │  value  │  units  │
│ varchar │ varchar │ varchar │  varchar  │ varchar │       varchar        │ varchar │            varchar            │       varchar        │                                                        varchar                                                        │ varchar │ varchar │
├─────────┼─────────┼─────────┼───────────┼─────────┼──────────────────────┼─────────┼───────────────────────────────┼──────────────────────┼─────────────

In [22]:
bs_rel_normalized = duckdb.sql("""
        SELECT
            CAST(period  || '-01' AS TIMESTAMP)   AS period
            ,duoarea
            ,"area-name"                AS "area_name"
            ,product
            ,"product-name"             AS "product_name"
            ,process         
            ,"process-name"             AS "process_name"
            ,series
            ,"series-description"       AS "series_description"
            ,TRY_CAST(value AS DOUBLE)  AS value
            ,units
        FROM bs_rel
    """)

In [ ]:
# Define the schema for the PostgreSQL table
write_to_postgres(
    relation=bs_rel_normalized,
    table_name="t_supply_and_disposition",
    schema={
        "period":               ("TIMESTAMP", True),
        "duoarea":              ("TEXT", False),
        "area_name":            ("TEXT", True),
        "product":              ("TEXT", True),
        "product_name":         ("TEXT", False),
        "process":              ("TEXT", True),
        "process_name":         ("TEXT", False),
        "series":               ("TEXT", True),
        "series_description":   ("TEXT", False),
        "value":                ("NUMERIC", False),
        "units":                ("TEXT", False)
    },
    engine=engine,
)

Schema 'bronze' is ready.
[INFO] Created 'bronze.t_supply_and_disposition' with PKs: ['period', 'area_name', 'product', 'process', 'series']
[INFO] Upserted 1013282 records into 'bronze.t_supply_and_disposition' on PK conflict of ['period', 'area_name', 'product', 'process', 'series'].


In [ ]:
# Clear variables to free memory before next fetch
del bs_rel, bs_rel_normalized

#### petroleum/sum/sndw

(Weekly Supply and Disposition)

In [65]:
eia_get_route_metadata("petroleum/sum/sndw")

{'facets': [{'id': 'duoarea', 'description': 'DuoArea'},
  {'id': 'product', 'description': 'Product'},
  {'id': 'process', 'description': 'Process'},
  {'id': 'series', 'description': 'Series'}],
 'data_cols': ['value'],
 'frequencies': []}

In [68]:
cols = eia_get_data_columns("petroleum/sum/sndw")
print(cols) # CHECK

['value']


In [70]:
params = eia_build_params(
    route="petroleum/sum/sndw",
    frequency=None,
    start="2000-01",
    end="2025-12",
    data_cols=['value'],
    facets={
        "duoarea": None,
        "product": None,
        "process": None,
        "series": None
    },
)

sndw_rel = eia_fetch_all("petroleum/sum/sndw", params)


[INFO] Discovered 23 values for facet 'duoarea': ['R20', 'R4N5', 'YCUOK', 'R1Z', 'R4N5-Z00', 'R1X-Z00', 'R10-Z00', 'R30-Z00', 'R50-Z00', 'R30', 'NUS-Z00', 'R10', 'R20-Z00', 'R40-Z00', 'R50', 'R1Y', 'SAK', 'NUS', 'R40', 'R1X', 'R1Y-Z00', 'R1Z-Z00', 'R48']
[INFO] Discovered 43 values for facet 'product': ['EPC0', 'EPM0R', 'EPPO4', 'EPD0', 'EPM0', 'EPJKM', 'EPOBGRRA', 'EPM0RO', 'EPLLPYN', 'EPOBG', 'EPDM10', 'EPDM20', 'EPD00H', 'EPM0RA', 'EPM0CO', 'EPOBGCO', 'EPLLPZ', 'EPM0CAL55', 'EPPK', 'EPM0RN', 'EPOBGRRE', 'EPOBGCC', 'EPOBGCG', '(NA)', 'EPP0', 'EPM0CAG55', 'EPOBGRR', 'EPOOXE', 'EPDXL0', 'EPDXH0', 'EPJKC', 'EPPR', 'EPPA', 'EPM0CA', 'EP00', 'EPL0XP', 'EPPO6', 'EPM0C', 'EPM0F', 'EPXXX2', 'EPJK', 'EPPU', 'EPLLP0C']
[INFO] Discovered 25 values for facet 'process': ['IMN', 'YOP', 'SAXP', 'SAXL', 'IM0', 'FPF', 'YPA', 'YUP', 'YIR', 'YPR', 'YPT', 'IMX', 'EEX', 'IMS', 'VPP', 'SKB', 'IMU', 'VUA', 'SAE', 'SAS', 'SKA', 'SAX', 'YIY', 'YRL', 'VSD']
[INFO] Discovered 608 values for facet 'series': ['W

In [73]:
print(sndw_rel.describe())

┌─────────┬────────────┬─────────┬───────────┬─────────┬──────────────────────┬─────────┬───────────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┐
│  aggr   │   period   │ duoarea │ area-name │ product │     product-name     │ process │   process-name    │        series        │                                           series-description                                           │  value  │  units  │
│ varchar │  varchar   │ varchar │  varchar  │ varchar │       varchar        │ varchar │      varchar      │       varchar        │                                                varchar                                                 │ varchar │ varchar │
├─────────┼────────────┼─────────┼───────────┼─────────┼──────────────────────┼─────────┼───────────────────┼──────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────────

In [74]:
sndw_rel_normalized = duckdb.sql("""
        SELECT
            period
            ,duoarea
            ,"area-name"                AS "area_name"
            ,product
            ,"product-name"             AS "product_name"
            ,process         
            ,"process-name"             AS "process_name"
            ,series
            ,"series-description"       AS "series_description"
            ,TRY_CAST(value AS DOUBLE)  AS value
            ,units
        FROM sndw_rel
    """)

In [75]:
# Define the schema for the PostgreSQL table
write_to_postgres(
    relation=sndw_rel_normalized,
    table_name="t_weekly_supply_and_disposition",
    schema={
        "period":               ("TEXT", True),
        "duoarea":              ("TEXT", False),
        "area_name":            ("TEXT", True),
        "product":              ("TEXT", True),
        "product_name":         ("TEXT", False),
        "process":              ("TEXT", True),
        "process_name":         ("TEXT", False),
        "series":               ("TEXT", True),
        "series_description":   ("TEXT", False),
        "value":                ("NUMERIC", False),
        "units":                ("TEXT", False)
    },
    engine=engine,
)

Schema 'bronze' is ready.
[INFO] Created 'bronze.t_weekly_supply_and_disposition' with PKs: ['period', 'area_name', 'product', 'process', 'series']
[INFO] Upserted 644954 records into 'bronze.t_weekly_supply_and_disposition' on PK conflict of ['period', 'area_name', 'product', 'process', 'series'].


#### natural-gas/prod/ngpl

(Natural Gas Plant Liquids)

In [45]:
eia_get_route_metadata("natural-gas/prod/ngpl")

{'facets': [{'id': 'duoarea', 'description': 'DuoArea'},
  {'id': 'product', 'description': 'Product'},
  {'id': 'process', 'description': 'Process'},
  {'id': 'series', 'description': 'Series'}],
 'data_cols': ['value'],
 'frequencies': []}

In [35]:
cols = eia_get_data_columns("natural-gas/prod/ngpl/")
print(cols) # CHECK

['value']


In [53]:
params = eia_build_params(
    route="natural-gas/prod/ngpl",
    frequency=None,
    start="2000-01",
    end="2025-12",
    data_cols=['value'],
    facets={
        "duoarea": None,
        "product": None,
        "process": None,
        "series": None
    },
)

ngpl_rel = eia_fetch_all("natural-gas/prod/ngpl", params)


[INFO] Discovered 49 values for facet 'duoarea': ['SOH', 'RTX01', 'R44F', 'RTX07B', 'RNMWEST', 'RTX03', 'RTX05', 'RTXSF', 'SCA', 'SND', 'SNM', 'SUT', 'SWY', 'RTX08A', 'RTX08', 'SKS', 'SKY', 'RCAC', 'RLASO', 'RTX07C', 'SAL', 'SAR', 'SFL', 'SMI', 'SMS', 'SWV', 'RUSF', 'RCAL', 'RCASF', 'RLASF', 'RTX09', 'SAK', 'SCO', 'SMT', 'SOK', 'NUS', 'R1901F', 'R5F', 'RNMEAST', 'RTX06', 'RUTWY', 'R48', 'RCAJ', 'RLAN', 'RTX02', 'RTX04', 'RTX10', 'SLA', 'STX']
[INFO] Discovered 1 values for facet 'product': ['EPL0']
[INFO] Discovered 1 values for facet 'process': ['R55']
[INFO] Discovered 49 values for facet 'series': ['RL2R55R48_1', 'RL2R55RTX02_1', 'RL2R55RTXSF_1', 'RL2R55SLA_1', 'RL2R55SND_1', 'RL2R55SNM_1', 'RL2R55RNMEAST_1', 'RL2R55RTX03_1', 'RL2R55RTX07B_1', 'RL2R55RTX07C_1', 'RL2R55RUTWY_1', 'RL2R55SKS_1', 'RES_EPL0_R55_SOH_MMBBL', 'RL2R55RCASF_1', 'RL2R55RLAN_1', 'RL2R55SFL_1', 'RL2R55SKY_1', 'RL2R55SMI_1', 'RL2R55SOK_1', 'RL2R55R1901F_1', 'RL2R55R5F_1', 'RL2R55RTX01_1', 'RL2R55RTX05_1', 'RL2R55

In [54]:
print(ngpl_rel.describe())

┌─────────┬─────────┬─────────┬────────────┬─────────┬───────────────────────────────────────────────┬─────────┬──────────────────────────────────────────────┬────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┐
│  aggr   │ period  │ duoarea │ area-name  │ product │                 product-name                  │ process │                 process-name                 │         series         │                                          series-description                                          │  value  │  units  │
│ varchar │ varchar │ varchar │  varchar   │ varchar │                    varchar                    │ varchar │                   varchar                    │        varchar         │                                               varchar                                                │ varchar │ varchar │
├─────────┼─────────┼─────────┼────────────┼─────────┼──────────────────────

In [61]:
ngpl_rel_normalized = duckdb.sql("""
        SELECT
            period
            ,duoarea
            ,"area-name"                AS "area_name"
            ,product
            ,"product-name"             AS "product_name"
            ,process         
            ,"process-name"             AS "process_name"
            ,series
            ,"series-description"       AS "series_description"
            ,TRY_CAST(value AS DOUBLE)  AS value
            ,units
        FROM ngpl_rel
    """)

In [64]:
# Define the schema for the PostgreSQL table
write_to_postgres(
    relation=ngpl_rel_normalized,
    table_name="t_natural_gas_plant_liquids",
    schema={
        "period":               ("TEXT", True),
        "duoarea":              ("TEXT", False),
        "area_name":            ("TEXT", True),
        "product":              ("TEXT", True),
        "product_name":         ("TEXT", False),
        "process":              ("TEXT", True),
        "process_name":         ("TEXT", False),
        "series":               ("TEXT", True),
        "series_description":   ("TEXT", False),
        "value":                ("NUMERIC", False),
        "units":                ("TEXT", False)
    },
    engine=engine,
)

Schema 'bronze' is ready.
[INFO] Created 'bronze.t_natural_gas_plant_liquids' with PKs: ['period', 'area_name', 'product', 'process', 'series']
[INFO] Upserted 1029 records into 'bronze.t_natural_gas_plant_liquids' on PK conflict of ['period', 'area_name', 'product', 'process', 'series'].


#### natural-gas/prod/sum

(Natural Gas Production Summary)

In [76]:
params = eia_build_params(
    route="natural-gas/prod/sum",
    frequency="monthly",
    start="2000-01",
    end="2025-12",
    facets={
        "duoarea": None,
        "product": None,
        "process": None,
        "series": None
    }
)

ngpl_s_rel = eia_fetch_all("natural-gas/prod/sum", params)


[INFO] Discovered 37 values for facet 'duoarea': ['SNE', 'SNY', 'SOH', 'SID', 'SIL', 'SOR', 'SCA', 'SND', 'SNM', 'SNV', 'SUT', 'SWY', 'SKS', 'SKY', 'SSD', 'SVA', 'R98', 'SAL', 'SAR', 'SFL', 'SMI', 'SMS', 'SWV', 'SAK', 'SCO', 'R3FM', 'SMD', 'SMT', 'SOK', 'STN', 'NUS', 'SAZ', 'SIN', 'SLA', 'SMO', 'SPA', 'STX']
[INFO] Discovered 1 values for facet 'product': ['EPG0']
[INFO] Discovered 11 values for facet 'process': ['FGW', 'VRN', 'VGQ', 'FPD', 'VGV', 'VGM', 'FGC', 'FGS', 'FGG', 'FGO', 'VG9']
[INFO] Discovered 434 values for facet 'series': ['N9010MD2', 'N9010MS2', 'N9010NY2', 'N9010OR2', 'N9010TN2', 'N9011MD2', 'N9011ND2', 'N9011OH2', 'N9012ND2', 'N9012TN2', 'N9012WY2', 'N9020AR2', 'N9020CO2', 'N9020KY2', 'N9020MO2', 'N9020US2', 'N9020UT2', 'N9020WY2', 'N9030AL2', 'N9030MD2', 'N9030NM2', 'N9030TX2', 'N9030WY2', 'N9040FL2', 'N9040FX2', 'N9040NY2', 'N9040OR2', 'N9050AL2', 'N9050MI2', 'N9050US2', 'N9050UT2', 'N9050WY2', 'NA1150_SAK_2', 'NA1150_SAR_2', 'NA1160_R3FM_2', 'NA1160_SAL_2', 'NA1160

In [77]:
print(ngpl_s_rel.describe())

┌─────────┬─────────┬─────────┬────────────┬─────────┬──────────────┬─────────┬────────────────────────────┬───────────────────────┬──────────────────────────────────────────────────────────────┬─────────┬─────────┐
│  aggr   │ period  │ duoarea │ area-name  │ product │ product-name │ process │        process-name        │        series         │                      series-description                      │  value  │  units  │
│ varchar │ varchar │ varchar │  varchar   │ varchar │   varchar    │ varchar │          varchar           │        varchar        │                           varchar                            │ varchar │ varchar │
├─────────┼─────────┼─────────┼────────────┼─────────┼──────────────┼─────────┼────────────────────────────┼───────────────────────┼──────────────────────────────────────────────────────────────┼─────────┼─────────┤
│ count   │ 121148  │ 121148  │ 121148     │ 121148  │ 121148       │ 121148  │ 121148                     │ 121148                │ 121

In [ ]:
ngpl_s_rel_normalized = duckdb.sql("""
        SELECT
            CAST(period  || '-01' AS TIMESTAMP)   AS period
            ,duoarea
            ,"area-name"                AS "area_name"
            ,product
            ,"product-name"             AS "product_name"
            ,process         
            ,"process-name"             AS "process_name"
            ,series
            ,"series-description"       AS "series_description"
            ,TRY_CAST(value AS DOUBLE)  AS value
            ,units
        FROM ngpl_s_rel
    """)

In [80]:
# Define the schema for the PostgreSQL table
write_to_postgres(
    relation=ngpl_s_rel,
    table_name="t_natural_gas_plant_liquids_summary",
    schema={
        "period":               ("TIMESTAMP", True),
        "duoarea":              ("TEXT", False),
        "area_name":            ("TEXT", True),
        "product":              ("TEXT", True),
        "product_name":         ("TEXT", False),
        "process":              ("TEXT", True),
        "process_name":         ("TEXT", False),
        "series":               ("TEXT", True),
        "series_description":   ("TEXT", False),
        "value":                ("NUMERIC", False),
        "units":                ("TEXT", False)
    },
    engine=engine,
)

Schema 'bronze' is ready.
[INFO] Created 'bronze.t_natural_gas_plant_liquids_summary' with PKs: ['period', 'area_name', 'product', 'process', 'series']
[INFO] Upserted 121148 records into 'bronze.t_natural_gas_plant_liquids_summary' on PK conflict of ['period', 'area_name', 'product', 'process', 'series'].


#### WTI Crude Spot Price (daily)

In [ ]:
# --- WTI Crude Spot Price (daily) ---
params = {
    "frequency": "daily",
    "data[0]": "value",
    "facets[series][]": "RWTC",
    "start": "2015-01-01",
    "end": "2024-12-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_wti = eia_fetch("petroleum/pri/spt", params)
display(df_wti)

#### Natural Gas Spot Price (Henry Hub, daily)

In [ ]:
# --- Natural Gas Spot Price (Henry Hub, daily) ---
params = {
    "frequency": "daily",
    "data[0]": "value",
    "facets[series][]": "RNGWHHD",
    "start": "2015-01-01",
    "end": "2024-12-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_ng_price = eia_fetch("natural-gas/pri/sum", params)
display(df_ng_price)

#### Weekly Crude Inventory (US Total Stocks)

In [ ]:
# --- Weekly Crude Inventory (US total stocks) ---
params = {
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "WTTSTUS1",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_inventory = eia_fetch("petroleum/stoc/wstk", params)
display(df_inventory)